In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# linear regression using statsmodels
import statsmodels.api as sm
from statsmodels.formula.api import ols

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
# from scipy.stats import t

from sklearn.metrics import mean_squared_error

In [ ]:
# importing data

bike_data = pd.read_csv('./data/daily-bike-share.csv')
bike_data.head()

## Multivariate linear regression

In [ ]:
f = "rentals ~ temp + windspeed"
m1_statsmodel = ols(formula = f, data = bike_data).fit()

## Standard Errors of Parameters

In [ ]:
## Step by Step version

from sklearn.metrics import mean_absolute_error

# features = ['temp']
features = ['temp', 'windspeed']
target = 'rentals'

# X = bike_data[features].to_numpy() is equivalent
X = bike_data[features].values
y = bike_data[target].values

m1_sklearn = LinearRegression().fit(
    X = X,
    y = y
)
# n: number of datapoints
n = bike_data.shape[0]
# m: number of features
m = len(features)

# Creating Design Matrix - first column is the intercept
X_design = np.hstack([np.ones(n).reshape(-1,1), X])
# beta is the array of the estimated parameters
       
beta = np.concatenate((np.array([m1_sklearn.intercept_]), m1_sklearn.coef_))
# the predicted output based on the estimated parameters can be computed as
# y_hat = X_design @ beta.reshape(-1,1)
# or using the predict function of sklearn LinearRegression
y_hat = m1_sklearn.predict(X)

# Compute Residual Standard Error (RSE)
# residuals
residuals = y - y_hat
k = m + 1         # Number of parameters (intercept + slope)
SSR = np.sum(residuals**2)  # Sum of squared residuals
RSE = np.sqrt(SSR / (n - k))  # Residual Standard Error

# Compute the variance-covariance matrix
cov_matrix = RSE**2 * np.linalg.inv(X_design.T @ X_design)

# Step 6: Extract standard errors (sqrt of diagonal elements)
bse = np.sqrt(np.diag(cov_matrix))

In [ ]:
m1_sklearn.coef_[0:]

In [ ]:
def params_std_err(data, features, target, alpha = 0.05):
    # X = bike_data[features].to_numpy() is equivalent
    X = data[features].values
    y = data[target].values

    model = LinearRegression().fit(
        X = X,
        y = y
    )

    # n: number of datapoints
    n = data.shape[0]
    # m: number of features
    m = len(features)

    # Creating Design Matrix - first column is the intercept
    X_design = np.hstack([np.ones(n).reshape(-1,1), X])
    # beta is the array of the estimated parameters
#    return(model)
    beta = np.concatenate((np.array([model.intercept_]), model.coef_))
    # the predicted output based on the estimated parameters can be computed as
    # y_hat = X_design @ beta.reshape(-1,1)
    # or using the predict function of sklearn LinearRegression
    y_hat = model.predict(X)
    # Compute Residual Standard Error (RSE)
    # residuals
    residuals = y - y_hat
    k = m + 1         # Number of parameters (intercept + slope)
    SSR = np.sum(residuals**2)  # Sum of squared residuals
    RSE = np.sqrt(SSR / (n - k))  # Residual Standard Error
    
    # Compute the variance-covariance matrix
    cov_matrix = RSE**2 * np.linalg.inv(X_design.T @ X_design)
    
    # Step 6: Extract standard errors (sqrt of diagonal elements)
    bse = np.sqrt(np.diag(cov_matrix))
    return(
        pd.DataFrame(
            {
                'param' : ['Intercept'] + features,
                'coef' : beta,
                'std err' : bse
            }
        )
    )

In [ ]:
mtest = params_std_err(bike_data, 
               ['temp', 'windspeed'], 
               'rentals'
              )

In [ ]:
mtest

In [ ]:
m1_statsmodel.summary()

## Prediction Confidence Intervals and Prediction Intervals

In [ ]:
x_new = pd.DataFrame(
    {
        'temp' : np.array([0.2, 0.6]),
        'windspeed' : np.array([0.3, 0.4])
    }
)

x_new

In [ ]:
predictions = m1_statsmodel.get_prediction(x_new)
predictions.summary_frame(alpha = 0.05)

In [ ]:
m1_sklearn.coef_

In [ ]:
m1_sklearn.predict(x_new[['temp', 'windspeed']].values)

In [ ]:
## Prediction Step by Step

alpha = 0.05

features = ['temp', 'windspeed']
target = 'rentals'

## Points prediction

x_new = pd.DataFrame(
    {
        'temp' : np.array([0.2, 0.6]),
        'windspeed' : np.array([0.3, 0.4])
    }
)

## Prediction Matrix Design

n_predictions = x_new.shape[0]
X_new_design = np.hstack([np.ones(n_predictions).reshape(-1,1), x_new.values])

# X = bike_data[features].to_numpy() is equivalent
X = bike_data[features].values
y = bike_data[target].values

model_test = LinearRegression().fit(
    X = X,
    y = y
)
# n: number of datapoints
n = bike_data.shape[0]
# m: number of features
m = len(features)

# Creating Design Matrix - first column is the intercept
X_design = np.hstack([np.ones(n).reshape(-1,1), X])
# beta is the array of the estimated parameters
       
beta = np.concatenate((np.array([m1_sklearn.intercept_]), m1_sklearn.coef_))

y_hat = model_test.predict(X)
# Compute Residual Standard Error (RSE)
# residuals
residuals = y - y_hat
k = m + 1         # Number of parameters (intercept + slope)
SSR = np.sum(residuals**2)  # Sum of squared residuals
RSE = np.sqrt(SSR / (n - k))  # Residual Standard Error

# Compute (X'X)^(-1)
XtX_inv = np.linalg.inv(X_design.T @ X_design)

y_new_pred = model_test.predict(x_new.values)

# Compute CI and PI for multiple points
alpha = 0.05
t_value = stats.t.ppf(1 - alpha / 2, n - m - 1)  # t-critical value

# Compute CI and PI margins for each new observation
ci_margins = t_value * RSE * np.sqrt(np.einsum('ij,ji->i', X_new_design @ XtX_inv, X_new_design.T))
pi_margins = t_value * RSE * np.sqrt(1 + np.einsum('ij,ji->i', X_new_design @ XtX_inv, X_new_design.T))

In [ ]:
ci_margins

In [ ]:
pi_margins

In [ ]:
predictions.summary_frame(alpha = 0.05)

In [ ]:
y_new_pred

In [ ]:
RSE

In [ ]:
RSE

In [ ]:
np.sqrt(mean_squared_error(y, y_hat))

In [ ]:
((y_hat - y)**2).sum()/(len(y)-2)

In [ ]:
mean_squared_error(y_hat, y)

In [ ]:
((y_hat - y)**2).sum()/(len(y)-2)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import t

np.set_printoptions(precision=10, suppress=True)

# Example data
bike_data = pd.DataFrame({
    'temp': [0.1, 0.2, 0.3, 0.4, 0.5],
    'humidity': [80, 70, 60, 50, 40],
    'windspeed': [5, 10, 15, 20, 25],
    'rentals': [100, 200, 300, 400, 500]
})

# Features (X) and target (y)
X = bike_data[['temp', 'humidity', 'windspeed']].values
y = bike_data['rentals'].values

# Add constant for intercept (Design Matrix)
X_design = np.column_stack((np.ones(X.shape[0]), X))

# Fit model using sklearn
model = LinearRegression().fit(X, y)

# Predicted values
y_pred = model.predict(X)

# Residual standard error (RSE)
n, p = X.shape  # n = number of data points, p = number of predictors
mse = mean_squared_error(y, y_pred)
s = np.sqrt(mse)  # Residual standard error

# Compute (X'X)^(-1)
XtX_inv = np.linalg.inv(X_design.T @ X_design)

# New data for prediction (multiple rows)
X_new = np.array([
    [1, 0.5, 45, 12],    # First observation
    [1, 0.3, 60, 10]     # Second observation
])  # Include 1 for intercept

# Predict new values
y_new_pred = model.predict(X_new[:, 1:])  # Exclude the intercept column

# Compute CI and PI for multiple points
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, n - p - 1)  # t-critical value

# Compute CI and PI margins for each new observation
# ci_margins = t_value * s * np.sqrt(np.sum(X_new @ XtX_inv * X_new, axis=1))
# ci_margins = t_value * s * np.sqrt(np.einsum('ij,ji->i', X_new @ XtX_inv, X_new.T))
# pi_margins = t_value * s * np.sqrt(1 + np.einsum('ij,ji->i', X_new @ XtX_inv, X_new.T))


ci_margins = t_value * s * np.sqrt(np.sum(X_new @ XtX_inv * X_new, axis=1))
pi_margins = t_value * s * np.sqrt(1 + np.sum(X_new @ XtX_inv * X_new, axis=1))


# Confidence Intervals
ci_lower = y_new_pred - ci_margins
ci_upper = y_new_pred + ci_margins

# Prediction Intervals
pi_lower = y_new_pred - pi_margins
pi_upper = y_new_pred + pi_margins

# Results
for i in range(len(y_new_pred)):
    print(f"Prediction {i+1}: {y_new_pred[i]:.3f}")
    print(f"95% Confidence Interval: ({ci_lower[i]:.3f}, {ci_upper[i]:.3f})")
    print(f"95% Prediction Interval: ({pi_lower[i]:.3f}, {pi_upper[i]:.3f})\n")

In [ ]:
X_new

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.stats import t

# Example data
bike_data = pd.DataFrame({
    'temp': [0.1, 0.2, 0.3, 0.4, 0.5],
    'humidity': [80, 70, 60, 50, 40],
    'windspeed': [5, 10, 15, 20, 25],
    'rentals': [100, 200, 300, 400, 500]
})

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_new_scaled = scaler.transform(X_new[:, 1:])  # Exclude intercept for scaling

X_design = np.column_stack((np.ones(X_scaled.shape[0]), X_scaled))
X_new = np.column_stack((np.ones(X_new_scaled.shape[0]), X_new_scaled))

# Features (X) and target (y)
X = bike_data[['temp', 'humidity', 'windspeed']].values
y = bike_data['rentals'].values

# Add constant for intercept (Design Matrix)
X_design = np.column_stack((np.ones(X.shape[0]), X))

# Fit model using sklearn
model = LinearRegression().fit(X, y)

# Predicted values
y_pred = model.predict(X)

# Residual standard error (RSE)
n, p = X.shape  # n = number of data points, p = number of predictors
mse = mean_squared_error(y, y_pred)
s = np.sqrt(mse)  # Residual standard error

# Compute (X'X)^(-1)
XtX_inv = np.linalg.pinv(X_design.T @ X_design)

# New data for prediction (multiple rows)
X_new = np.array([
    [1, 0.5, 45, 12],    # First observation
    [1, 0.3, 60, 10]     # Second observation
])  # Include 1 for intercept

# Predict new values
y_new_pred = model.predict(X_new[:, 1:])  # Exclude the intercept column

# Compute CI and PI for multiple points
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, n - p - 1)  # t-critical value

# Corrected calculation
ci_margins = t_value * s * np.sqrt(np.sum(X_new @ XtX_inv * X_new, axis=1))
pi_margins = t_value * s * np.sqrt(1 + np.sum(X_new @ XtX_inv * X_new, axis=1))

# Confidence Intervals
ci_lower = y_new_pred - ci_margins
ci_upper = y_new_pred + ci_margins

# Prediction Intervals
pi_lower = y_new_pred - pi_margins
pi_upper = y_new_pred + pi_margins

# Results
for i in range(len(y_new_pred)):
    print(f"Prediction {i+1}: {y_new_pred[i]:.3f}")
    print(f"95% Confidence Interval: ({ci_lower[i]:.3f}, {ci_upper[i]:.3f})")
    print(f"95% Prediction Interval: ({pi_lower[i]:.3f}, {pi_upper[i]:.3f})\n")

In [ ]:
print(f"X_design:\n{X_design}")
print(f"X_new:\n{X_new}")
print(f"XtX_inv:\n{XtX_inv}")
print(f"Residual Standard Error (s): {s}")
print(f"Degrees of Freedom: {n - p - 1}")
print(f"t-value: {t_value}")
print(f"ci_margins: {ci_margins}")
print(f"pi_margins: {pi_margins}")

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import t

# Original data
X = np.array([
    [0.1, 80, 5],
    [0.2, 70, 10],
    [0.3, 60, 15],
    [0.4, 50, 20],
    [0.5, 40, 25]
])
y = np.array([450, 400, 350, 300, 250])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Add intercept term
X_design = np.column_stack((np.ones(X_scaled.shape[0]), X_scaled))

# Fit model
model = LinearRegression().fit(X_design, y)

# New points for prediction
X_new = np.array([
    [0.5, 45, 12],
    [0.3, 60, 10]
])
X_new_scaled = scaler.transform(X_new)
X_new_design = np.column_stack((np.ones(X_new_scaled.shape[0]), X_new_scaled))

# Predictions
y_pred = model.predict(X_new_design)

# Residual Standard Error (s)
y_hat = model.predict(X_design)
residuals = y - y_hat
n, p = X_design.shape
s = np.sqrt(np.sum(residuals**2) / (n - p))

# Inverted (X'X) using pseudo-inverse for stability
XtX_inv = np.linalg.pinv(X_design.T @ X_design)

# Compute Confidence & Prediction Intervals
t_value = t.ppf(0.975, n - p)

ci_margins = t_value * s * np.sqrt(np.sum(X_new_design @ XtX_inv * X_new_design, axis=1))
pi_margins = t_value * s * np.sqrt(1 + np.sum(X_new_design @ XtX_inv * X_new_design, axis=1))

# Results
for i in range(len(X_new)):
    print(f"Prediction {i+1}: {y_pred[i]:.3f}")
    print(f"95% Confidence Interval: ({y_pred[i] - ci_margins[i]:.3f}, {y_pred[i] + ci_margins[i]:.3f})")
    print(f"95% Prediction Interval: ({y_pred[i] - pi_margins[i]:.3f}, {y_pred[i] + pi_margins[i]:.3f})\n")


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import t

# Original data
X = np.array([
    [0.1, 80, 5],
    [0.2, 70, 10],
    [0.3, 60, 15],
    [0.4, 50, 20],
    [0.5, 40, 25]
])
y = np.array([450, 400, 350, 300, 250])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Add intercept term
X_design = np.column_stack((np.ones(X_scaled.shape[0]), X_scaled))

# Fit model
model = LinearRegression().fit(X_design, y)

# New points for prediction
X_new = np.array([
    [0.5, 45, 12],
    [0.3, 60, 10]
])
X_new_scaled = scaler.transform(X_new)
X_new_design = np.column_stack((np.ones(X_new_scaled.shape[0]), X_new_scaled))

# Predictions
y_pred = model.predict(X_new_design)

# Residual Standard Error (s)
y_hat = model.predict(X_design)
residuals = y - y_hat
n, p = X_design.shape
s = np.sqrt(np.sum(residuals**2) / (n - p))

# Inverted (X'X) using pseudo-inverse for stability
XtX_inv = np.linalg.pinv(X_design.T @ X_design)

# Compute Confidence & Prediction Intervals
t_value = t.ppf(0.975, n - p)

# Correct interval calculations
ci_margins = t_value * s * np.sqrt(np.sum(X_new_design @ XtX_inv * X_new_design.T, axis=1))
pi_margins = t_value * s * np.sqrt(1 + np.sum(X_new_design @ XtX_inv * X_new_design.T, axis=1))

# Results
for i in range(len(X_new)):
    print(f"Prediction {i+1}: {y_pred[i]:.3f}")
    print(f"95% Confidence Interval: ({y_pred[i] - ci_margins[i]:.3f}, {y_pred[i] + ci_margins[i]:.3f})")
    print(f"95% Prediction Interval: ({y_pred[i] - pi_margins[i]:.3f}, {y_pred[i] + pi_margins[i]:.3f})\n")


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import t

# Original data
X = np.array([
    [0.1, 80, 5],
    [0.2, 70, 10],
    [0.3, 60, 15],
    [0.4, 50, 20],
    [0.5, 40, 25]
])
y = np.array([450, 400, 350, 300, 250])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Add intercept term
X_design = np.column_stack((np.ones(X_scaled.shape[0]), X_scaled))

# Fit model
model = LinearRegression().fit(X_design, y)

# New points for prediction
X_new = np.array([
    [0.5, 45, 12],
    [0.3, 60, 10]
])
X_new_scaled = scaler.transform(X_new)
X_new_design = np.column_stack((np.ones(X_new_scaled.shape[0]), X_new_scaled))

# Predictions
y_pred = model.predict(X_new_design)

# Residual Standard Error (s)
y_hat = model.predict(X_design)
residuals = y - y_hat
n, p = X_design.shape
s = np.sqrt(np.sum(residuals**2) / (n - p))

# Inverted (X'X) using pseudo-inverse for stability
XtX_inv = np.linalg.pinv(X_design.T @ X_design)

# Compute Confidence & Prediction Intervals
t_value = t.ppf(0.975, n - p)

# Correct interval calculations using np.einsum() for row-wise dot product
ci_margins = t_value * s * np.sqrt(np.einsum('ij,ji->i', X_new_design, XtX_inv @ X_new_design.T))
pi_margins = t_value * s * np.sqrt(1 + np.einsum('ij,ji->i', X_new_design, XtX_inv @ X_new_design.T))

# Results
for i in range(len(X_new)):
    print(f"Prediction {i+1}: {y_pred[i]:.3f}")
    print(f"95% Confidence Interval: ({y_pred[i] - ci_margins[i]:.3f}, {y_pred[i] + ci_margins[i]:.3f})")
    print(f"95% Prediction Interval: ({y_pred[i] - pi_margins[i]:.3f}, {y_pred[i] + pi_margins[i]:.3f})\n")


In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

# Create a synthetic dataset
data = {
    'color': ['red', 'green', 'blue', 'red', 'green'],
    'size': [3, 5, 2, 4, 1],
    'price': [10, 20, 15, 25, 5]
}

df = pd.DataFrame(data)

# Separate features (X) and target variable (y)
X = df[['color', 'size']]
y = df['price']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One-Hot Encoding:
encoder_one_hot = OneHotEncoder()
X_train_one_hot = encoder_one_hot.fit_transform(X_train[['color']])

# Build linear regression model
model_one_hot = LinearRegression().fit(X_train_one_hot, y_train)

# Evaluate model on the test set
X_test_one_hot = encoder_one_hot.transform(X_test[['color']])
y_pred_one_hot = model_one_hot.predict(X_test_one_hot)
mse_one_hot = mean_squared_error(y_test, y_pred_one_hot)
print(f"One-Hot Encoding Model - Mean Squared Error: {mse_one_hot}")

One-Hot Encoding Model - Mean Squared Error: 225.0


In [2]:
X_train_one_hot

<4x3 sparse matrix of type '<class 'numpy.float64'>'
	with 4 stored elements in Compressed Sparse Row format>